# Implementation of the equations that make up the GSW functions for calculating the seawater surface density. A summary of the necessary steps are given in the cell below. 

# Mathematical equations

The **main equation** used is: 

$$
\hat{v} (S_A \Theta, p ) = v_u \sum_{i,j,k} v_{ijk} s^i \tau^j \pi^k  \tag{1}
$$

Where the values associated with i,j,k are given in the TEOS10 manual in Appendix K. 

$$
s = \sqrt{\frac{S_A + 24 gkg^-1}{S_{A_u}}}, \quad S_{A_u} = \frac{40 \times 35.16504 g kg^{-1}}{35}
$$

$$
\tau = \frac{\Theta}{\Theta_u}, \quad \Theta_u = 40^{\circ} C  
$$

$$
\pi = \frac{p}{p_u}, \quad P_u = 10^4 dbar
$$

$$
v_u = 1m^{3} kg^{-1}, \quad 
$$

However, p = 0 in this function and therefore $\pi \rightarrow 0$ when $k > 0$. Thats why the GSW function only needs salinity and temperature, not pressure for calculating the surface density: 

$$\hat{v} (S_A, \Theta, p = 0)$$


**Furthermore**, we also need to calculate the conserved temperature to use in the main equation. The following related equations are dedicated to this process.

For calculating the conserved temperature, we need to calculate both the entropy and a potential entalpi! The following steps are therefore needed in the calculations:

1. Calculation of the entropy (equation 2.10.1 from the TEOS manual): 

$$
\eta = \eta(S_A, t, p) = -g_T = - \frac{\partial g_T}{\partial T} \tag{2}
$$

In TEOS - empirical values are already used to create polynomial functions of $\eta$ - and I will rather use these estimated values directly with the aimn to reduce the need for computational calculations. These approximated values are found in appendix B of the TEOS and Copernicus manual! https://os.copernicus.org/articles/19/1719/2023/os-19-1719-2023.pdf

2. An iterativ calculation of potential temperature for the surface - ie. $\theta_0$. This is calculated using the Newton-Raphson iterative technique as illustrated in the equation below, found from TEOS10. It can also be estimated as an integral of the adiabatic lapse rate (Fofonoff, 1962 \& 1985), but I also suspect this would require potentially unneseccary computational power. 

$$
\eta (S_A, \theta, p_r) = \eta (S_A, t, p) \tag{3}
$$

The Newton Raphson iterative process is:
$$
x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)} \tag{4}
$$

Where we have to differentiate $g_t$ in our calculations. Furthermore, the full thermodynamic equation of entropy is given as: 

$$
\hat{\eta} (S_A, \Theta) = c_p^0 ln(1 + \Theta / T_0) + \alpha (\frac{S_A}{S_{SO}}) ln (\frac{S_A}{S_{SO}}) + P \{ 8,8 \} (s, \tau) \tag{5}
$$

Where: 

$$
T_0 = 273.15, \quad c_p^0 = 3991.86795711963 , \quad \alpha = - 9.309495003228781 , \quad S_{SO} = 35.16504 
$$

and the polynomial is differentiated such that the potency of $\tau$ \& $s$ is subtracted according to standard differentiation rules. 

3. Then we have to calculate the potential entalpy, where the reference pressure is always set to be zero because most heat flux activity is near the sea-surface. The reference pressure $p_r = 0$ dbar. This is calculated by using: 

$$
h^0 (S_A, t, p) = h (S_A, \theta, 0) = g(S_A, \theta, 0) - (T_0 + \theta)g_T (S_A, \theta, 0) \tag{6}
$$ 

Shortened to (I think - this is my doing hehe):
$$
h^0 = G - T * g_t 
$$

4. And yuhu now we can finally calculate the conserved temperature! Again with the use of the TEOS equations: 

$$
\Theta (S_A, t, p) = \tilde{\Theta} (S_A, \theta) = \frac{\tilde{h^0}(S_A, \theta)}{c_p^0} \tag{7}

$$

We have to differentiate the logarthitmic expression also. The second term $\rightarrow 0$, because it is independent of $\Theta$. So we are left with:

$$

\frac{\partial}{\partial \Theta} (c_p \cdot ln(1+ \frac{\Theta}{T_0})) =
(\frac{c_p}{T_0 + \Theta})'
$$


Potential entalpy

reference pressure = 0dbar = 1013hPa

Remember that $g_T$ is the Gibbs temperature derivated function, while $g = $ the non-derivated function.

$\eta = -g_T$

# Test blocks

In [1]:
from gsw_teos10_implementation import * 

In [2]:
#Test values
salinity = 33.5
Temp = 15

In [3]:
e_target, _ = entropy(SA = salinity, Theta = Temp)

In [4]:
density_field(SA = salinity, entropy_tar=e_target)

theta = 15.00000000000002, h0 = 59878.019356794524, Theta = 15.00000000000002
15.00000000000002
s = 1.2017813563920139, tau = 0.3750000000000005


(np.float64(1025.1193009880922), np.float64(25.119300988092164))

In [5]:
SA_test = 35.0
Theta_test = 15.0

eta_target, _ = entropy(SA_test, Theta_test)
print(f"Gitt: SA = {SA_test} g/kg, Theta_sann = {Theta_test} degC")
print(f"  -> entropi = {eta_target:.6f} J/(kg K)\n")

# Steg 1: entropi -> Theta (Newton-Raphson)
Theta_hat = theta_from_entropy(SA_test, eta_target)
print(f"[Steg 1] Theta fra Newton-Raphson       = {Theta_hat:.10f} degC "
      f"(avvik fra sann verdi: {Theta_hat - Theta_test:.2e} degC)")

# Steg 2: potensiell entalpi
h0 = potential_enthalpy(Theta_hat)
print(f"[Steg 2] potensiell entalpi h0           = {h0:.6f} J/kg")

# Steg 3: konservativ temperatur
CT_hat = conservative_T(h0)
print(f"[Steg 3] CT = h0/cp0                     = {CT_hat:.10f} degC\n")

print(f"Samlefunksjon CT_from_entropy()          = {CT(SA_test, eta_target):.10f} degC")

Gitt: SA = 35.0 g/kg, Theta_sann = 15.0 degC
  -> entropi = 213.440395 J/(kg K)

[Steg 1] Theta fra Newton-Raphson       = 15.0000000000 degC (avvik fra sann verdi: -2.31e-14 degC)
[Steg 2] potensiell entalpi h0           = 59878.019357 J/kg
[Steg 3] CT = h0/cp0                     = 15.0000000000 degC

theta = 14.999999999999977, h0 = 59878.01935679436, Theta = 14.999999999999977
Samlefunksjon CT_from_entropy()          = 15.0000000000 degC


In [6]:
v_hat = coeff_75term_polynomial(SA_test, eta_target)
print(f'v hat{v_hat}')
d = density_field(SA = salinity, entropy_tar=e_target)
print(f'Density field : {d}')

theta = 14.999999999999977, h0 = 59878.01935679436, Theta = 14.999999999999977
14.999999999999977
s = 1.2173558465554997, tau = 0.37499999999999944
v hat0.0009744001750633938
theta = 15.00000000000002, h0 = 59878.019356794524, Theta = 15.00000000000002
15.00000000000002
s = 1.2017813563920139, tau = 0.3750000000000005
Density field : (np.float64(1025.1193009880922), np.float64(25.119300988092164))


In [7]:
print(1 / v_hat) 

1026.2723936137847


In [8]:
import gsw

In [9]:
gsw.sigma0(SA_test, Theta_test)

np.float64(25.847622175753486)